In [ ]:
# Installs PyRadiomics from GitHub, connects Drive and writes the settings file (run once after every runtime reset).
!pip -q install "git+https://github.com/AIM-Harvard/pyradiomics.git"
from google.colab import drive
drive.mount('/content/drive')

with open('/content/pyradiomics_params.yaml', 'w') as f:
    f.write("""# PyRadiomics settings for M3-T03 (agreed with the team).
imageType:
  Original: {}
featureClass:
  shape:
  firstorder:
  glcm:
  glrlm:
  glszm:
  gldm:
  ngtdm:
setting:
  binWidth: 0.25
  normalize: false
  resampledPixelSpacing:
  label: 1
  force2D: false
""")

import sys, radiomics
print('Python', sys.version.split()[0], '| PyRadiomics', radiomics.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.5 MB/s eta 0:00:00
Mounted at /content/drive
Python 3.13.16 | PyRadiomics 3.1.1.dev111+g8ed579383


In [ ]:
# Copies the 188 validation cases (T1ce, FLAIR, T2 + predicted masks) to /content; files missing on Drive are taken from the trainval tar.
import os, shutil, tarfile
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

BASE = '/content/drive/MyDrive/BrainMRI_Data'
SRC = f'{BASE}/processed/brats2021'
TAR = f'{BASE}/processed/brats2021_trainval_v1.tar'
PRED_SRC = f'{BASE}/runs/M2-T04_nnunet/pred_nnunet_val'
OUT_IMG, OUT_PRED = '/content/radiomics/images', '/content/radiomics/pred'
MODS = ['t1ce', 'flair', 't2']

split = pd.read_csv(f'{BASE}/qc/split_v1.csv')
val = split[(split['dataset_source'] == 'brats2021') & (split['split'] == 'val')]
assert not val['lockbox'].astype(str).eq('True').any(), 'a locked patient is in the list'
val_ids = sorted(val['canonical_subject_id'].str.split('__').str[-1])
print('Validation patients:', len(val_ids))

os.makedirs(OUT_PRED, exist_ok=True)
jobs = [(f'{SRC}/{p}/0/{m}.nii.gz', f'{OUT_IMG}/{p}/{m}.nii.gz') for p in val_ids for m in MODS]
jobs += [(f'{PRED_SRC}/{p}.nii.gz', f'{OUT_PRED}/{p}.nii.gz') for p in val_ids]


def copy_one(job):
    src, dst = job
    if os.path.exists(dst):
        return None
    if not os.path.exists(src):
        return job
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(src, dst)
    return None


with ThreadPoolExecutor(max_workers=8) as ex:
    missing = [j for j in tqdm(ex.map(copy_one, jobs), total=len(jobs), desc='copy') if j]

missing_masks = [j for j in missing if j[1].startswith(OUT_PRED)]
wanted = {(j[1].split('/')[-2], os.path.basename(j[1])) for j in missing if j[1].startswith(OUT_IMG)}
print('Not found on Drive: images', len(wanted), '| masks', len(missing_masks))

# Take the missing image files from the tar (one pass, stops when all are found).
found = set()
if wanted:
    with tarfile.open(TAR, 'r|', bufsize=16 * 1024 * 1024) as tar, \
         tqdm(total=os.path.getsize(TAR), unit='B', unit_scale=True, desc='tar') as bar:
        for m in tar:
            bar.update(m.size)
            if not m.isfile():
                continue
            name = os.path.basename(m.name)
            pid = next((p for p in m.name.split('/') if p.startswith('BraTS2021_')), None)
            if (pid, name) in wanted and (pid, name) not in found:
                dst = f'{OUT_IMG}/{pid}/{name}'
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with open(dst, 'wb') as out:
                    out.write(tar.extractfile(m).read())
                found.add((pid, name))
                if found == wanted:
                    break

n_img = sum(os.path.exists(f'{OUT_IMG}/{p}/{m}.nii.gz') for p in val_ids for m in MODS)
n_pred = sum(os.path.exists(f'{OUT_PRED}/{p}.nii.gz') for p in val_ids)
print(f'\nImages ready: {n_img} of {len(val_ids) * 3} | Masks ready: {n_pred} of {len(val_ids)}')
print('Still missing images:', sorted(wanted - found))
print('Missing masks:', [os.path.basename(j[1]) for j in missing_masks])

Validation patients: 188


copy:   0%|          | 0/752 [00:00<?, ?it/s]

Not found on Drive: images 54 | masks 0


tar:   0%|          | 0.00/15.6G [00:00<?, ?B/s]


Images ready: 564 of 564 | Masks ready: 188 of 188
Still missing images: []
Missing masks: []


In [ ]:
# Runs PyRadiomics on all 188 validation patients (WT, TC, ET x T1ce, FLAIR, T2), saves progress to Drive, and can resume if stopped.
import os, time, logging, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np, pandas as pd, SimpleITK as sitk
from tqdm.auto import tqdm
from radiomics import featureextractor

BASE = '/content/drive/MyDrive/BrainMRI_Data'
PARAMS = '/content/pyradiomics_params.yaml'
IMG_DIR, PRED_DIR = '/content/radiomics/images', '/content/radiomics/pred'
OUT_CSV = f'{BASE}/qc/radiomics_features_val.csv'
MODS = ['t1ce', 'flair', 't2']
REGIONS = {'WT': (1, 2, 3), 'TC': (1, 3), 'ET': (3,)}
WORKERS = os.cpu_count() or 2
assert os.path.exists(PARAMS), 'Run cell 1 first: it writes the settings file.'

_extractor = None


def to_float(value):
    # Shape features can come back as complex numbers; keep the real part, or NaN if the imaginary part is not zero.
    v = np.asarray(value).item()
    if isinstance(v, complex):
        return v.real if abs(v.imag) < 1e-9 else float('nan')
    return float(v)


def process_patient(pid):
    global _extractor
    logging.getLogger('radiomics').setLevel(logging.ERROR)
    if _extractor is None:
        _extractor = featureextractor.RadiomicsFeatureExtractor(PARAMS)
    rows = []
    try:
        pred = sitk.ReadImage(f'{PRED_DIR}/{pid}.nii.gz')
        pred_arr = sitk.GetArrayFromImage(pred)
        images = {m: sitk.ReadImage(f'{IMG_DIR}/{pid}/{m}.nii.gz') for m in MODS}
    except Exception as e:
        return [{'patient_id': pid, 'region': r, 'voxels': 0, 'skip_reason': f'input_unreadable:{type(e).__name__}'} for r in REGIONS]
    for region, labels in REGIONS.items():
        region_arr = np.isin(pred_arr, labels)
        row = {'patient_id': pid, 'region': region, 'voxels': int(region_arr.sum()), 'skip_reason': ''}
        if row['voxels'] == 0:
            row['skip_reason'] = 'empty_region'
            rows.append(row)
            continue
        mask = sitk.GetImageFromArray(region_arr.astype(np.uint8))
        mask.CopyInformation(pred)
        for mod in MODS:
            try:
                result = _extractor.execute(images[mod], mask)
            except Exception as e:
                row['skip_reason'] += f'{mod}:{type(e).__name__}:{str(e)[:80]} '
                continue
            for key, value in result.items():
                if key.startswith('diagnostics') or (key.startswith('original_shape') and mod != MODS[0]):
                    continue
                name = key.replace('original_', '')
                row[name if name.startswith('shape') else f'{mod}_{name}'] = to_float(value)
        rows.append(row)
    return rows


def save(rows):
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    pd.DataFrame(rows).sort_values(['patient_id', 'region']).to_csv(OUT_CSV, index=False)


split = pd.read_csv(f'{BASE}/qc/split_v1.csv')
val = split[(split['dataset_source'] == 'brats2021') & (split['split'] == 'val')]
assert not val['lockbox'].astype(str).eq('True').any(), 'a locked patient is in the list'
val_ids = sorted(val['canonical_subject_id'].str.split('__').str[-1])

rows = pd.read_csv(OUT_CSV).to_dict('records') if os.path.exists(OUT_CSV) else []
todo = [p for p in val_ids if p not in {r['patient_id'] for r in rows}]
print(f'Validation patients: {len(val_ids)} | already done: {len(val_ids) - len(todo)} | to do: {len(todo)} | processes: {WORKERS}')

t0 = time.time()
with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context('fork')) as ex:
    futures = {ex.submit(process_patient, p): p for p in todo}
    for i, fut in enumerate(tqdm(as_completed(futures), total=len(futures)), 1):
        try:
            rows += fut.result()
        except Exception as e:
            rows += [{'patient_id': futures[fut], 'region': r, 'voxels': 0, 'skip_reason': f'worker_failed:{type(e).__name__}'} for r in REGIONS]
        if i % 10 == 0:
            save(rows)
save(rows)

out = pd.DataFrame(rows)
feat_cols = [c for c in out.columns if c not in ('patient_id', 'region', 'voxels', 'skip_reason')]
print(f'\nTime: {(time.time() - t0) / 60:.1f} min | Patients: {out.patient_id.nunique()} | Rows: {len(out)} | Feature columns: {len(feat_cols)}')
print('Rows with a skip reason:', int((out.skip_reason.fillna('') != '').sum()))
print(out[out.skip_reason.fillna('') != ''][['patient_id', 'region', 'voxels', 'skip_reason']].to_string(index=False))
print('Missing feature values (NaN):', int(out[feat_cols].isna().sum().sum()))
print('Saved to:', OUT_CSV)

Validation patients: 188 | already done: 10 | to do: 178 | processes: 2


  0%|          | 0/178 [00:00<?, ?it/s]


Time: 66.9 min | Patients: 188 | Rows: 564 | Feature columns: 293
Rows with a skip reason: 0
Empty DataFrame
Columns: [patient_id, region, voxels, skip_reason]
Index: []
Missing feature values (NaN): 0
Saved to: /content/drive/MyDrive/BrainMRI_Data/qc/radiomics_features_val.csv
